# Stage 2: Data Collection & Data Understanding

This notebook covers downloading, organizing, and understanding the MVTec AD dataset.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## 2.1 Dataset Structure

The MVTec AD dataset is organized as:
```
mvtec_anomaly_detection/
    bottle/
        train/good/          <- normal training images
        test/good/           <- normal test images
        test/broken_large/   <- defective test images
        test/broken_small/
        test/contamination/
        ground_truth/        <- pixel-level defect masks
    cable/
    ...(15 categories total)
```

In [ ]:
# ============================================================
# CONFIGURATION — Update DATA_ROOT to your dataset path
# ============================================================
DATA_ROOT = Path('../data/mvtec_anomaly_detection')
CATEGORY = 'bottle'  # Change to explore other categories

# All 15 MVTec categories
ALL_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

print(f'Dataset root: {DATA_ROOT}')
print(f'Selected category: {CATEGORY}')
print(f'All categories ({len(ALL_CATEGORIES)}): {ALL_CATEGORIES}')

In [ ]:
def count_images(data_root, category):
    """Count images per split and class for a given category."""
    cat_path = data_root / category
    stats = []
    
    for split in ['train', 'test']:
        split_path = cat_path / split
        if not split_path.exists():
            continue
        for defect_type in split_path.iterdir():
            if defect_type.is_dir():
                imgs = list(defect_type.glob('*.png')) + list(defect_type.glob('*.jpg'))
                label = 'normal' if defect_type.name == 'good' else 'defective'
                stats.append({
                    'split': split,
                    'defect_type': defect_type.name,
                    'label': label,
                    'count': len(imgs)
                })
    return pd.DataFrame(stats)

# Show dataset stats for selected category
if DATA_ROOT.exists():
    df_stats = count_images(DATA_ROOT, CATEGORY)
    print(f'\nDataset statistics for category: {CATEGORY}')
    print(df_stats.to_string(index=False))
    print(f'\nTotal images: {df_stats["count"].sum()}')
else:
    print('Dataset not found. Please download MVTec AD from:')
    print('https://www.mvtec.com/company/research/datasets/mvtec-ad')
    print('\nShowing expected structure...')
    # Demo stats (replace with real once dataset downloaded)
    demo = pd.DataFrame([
        {'split': 'train', 'defect_type': 'good', 'label': 'normal', 'count': 209},
        {'split': 'test', 'defect_type': 'good', 'label': 'normal', 'count': 83},
        {'split': 'test', 'defect_type': 'broken_large', 'label': 'defective', 'count': 20},
        {'split': 'test', 'defect_type': 'broken_small', 'label': 'defective', 'count': 21},
        {'split': 'test', 'defect_type': 'contamination', 'label': 'defective', 'count': 21},
    ])
    print(demo.to_string(index=False))

In [ ]:
def visualize_samples(data_root, category, n=4):
    """Visualize sample images from train (normal) and test (defective)."""
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    fig.suptitle(f'Sample Images — {category}', fontsize=16, fontweight='bold')
    
    # Normal images (train/good)
    normal_path = data_root / category / 'train' / 'good'
    normal_imgs = sorted(normal_path.glob('*.png'))[:n] if normal_path.exists() else []
    
    for i, img_path in enumerate(normal_imgs[:n]):
        img = Image.open(img_path).resize((224, 224))
        axes[0, i].imshow(img)
        axes[0, i].set_title('Normal', color='green', fontweight='bold')
        axes[0, i].axis('off')
    
    # Defective images (test/defect_type)
    test_path = data_root / category / 'test'
    defect_imgs = []
    if test_path.exists():
        for dtype in test_path.iterdir():
            if dtype.name != 'good':
                defect_imgs.extend(list(dtype.glob('*.png')))
    
    for i, img_path in enumerate(defect_imgs[:n]):
        img = Image.open(img_path).resize((224, 224))
        axes[1, i].imshow(img)
        axes[1, i].set_title(f'Defective\n({img_path.parent.name})', color='red', fontweight='bold')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig('../assets/eda_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

if DATA_ROOT.exists():
    visualize_samples(DATA_ROOT, CATEGORY)
else:
    print('Skipping visualization — dataset not downloaded yet')

## 2.2 Data Understanding Summary

Key observations:
- **Class imbalance:** Training set contains only normal images (unsupervised setting is also valid)
- **Multiple defect types per category:** broken_large, broken_small, contamination, etc.
- **High resolution:** Original images up to 1024×1024, resized to 224×224 for CNN
- **Ground truth masks:** Binary pixel masks available for all defective test images
- **Grayscale vs RGB:** Some categories are grayscale (grid, screw), most are RGB